# EDA — Datathon FIAP Fase 05
Análise exploratória dos tickets sintéticos da AmazoniaShop.


In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data" / "raw" / "tickets.csv")
print(df.shape)
df.head(3)

## 1. Distribuição de classes (category)

In [ ]:
cat_counts = df["category"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
cat_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Quantidade de tickets")
ax.set_title("Distribuição por categoria")
for i, v in enumerate(cat_counts):
    ax.text(v + 2, i, str(v), va="center")
plt.tight_layout()
plt.show()
print(cat_counts.to_string())

## 2. Distribuição por prioridade e canal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["priority"].value_counts().plot(kind="bar", ax=axes[0], color="coral", rot=0)
axes[0].set_title("Distribuição por prioridade")
axes[0].set_ylabel("Contagem")

df["channel"].value_counts().plot(kind="bar", ax=axes[1], color="mediumseagreen", rot=0)
axes[1].set_title("Distribuição por canal")
axes[1].set_ylabel("Contagem")

plt.tight_layout()
plt.show()

## 3. Comprimento médio da mensagem por categoria

In [ ]:
df["msg_len"] = df["message"].str.len()
avg_len = df.groupby("category")["msg_len"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
avg_len.plot(kind="barh", ax=ax, color="mediumpurple")
ax.set_xlabel("Comprimento médio (caracteres)")
ax.set_title("Comprimento médio da mensagem por categoria")
plt.tight_layout()
plt.show()
print(avg_len.round(1).to_string())

## 4. Top 10 palavras por categoria

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.features.feature_engineering import clean_text, STOPWORDS_PT

df["cleaned"] = df["message"].apply(clean_text)

categories = df["category"].unique()
n_cols = 3
n_rows = -(-len(categories) // n_cols)  # ceil division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes_flat = axes.flatten()

for idx, cat in enumerate(sorted(categories)):
    texts = df[df["category"] == cat]["cleaned"].str.cat(sep=" ")
    word_counts = pd.Series(texts.split()).value_counts().head(10)
    word_counts.plot(kind="barh", ax=axes_flat[idx], color="teal")
    axes_flat[idx].set_title(cat)
    axes_flat[idx].invert_yaxis()

for ax in axes_flat[len(categories):]:
    ax.set_visible(False)

plt.suptitle("Top 10 palavras por categoria", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Detecção de PII nos tickets

In [ ]:
_RE_EMAIL = re.compile(r"\S+@\S+\.\S+")
_RE_CPF   = re.compile(r"\d{3}\.?\d{3}\.?\d{3}-?\d{2}")
_RE_PHONE = re.compile(r"\(?\d{2}\)?\s?\d{4,5}-?\d{4}")

df["has_email"] = df["message"].str.contains(_RE_EMAIL, regex=True)
df["has_cpf"]   = df["message"].str.contains(_RE_CPF,   regex=True)
df["has_phone"] = df["message"].str.contains(_RE_PHONE, regex=True)
df["has_pii"]   = df[["has_email", "has_cpf", "has_phone"]].any(axis=1)

pii_pct = df["has_pii"].mean() * 100
print(f"Tickets com PII detectada: {df['has_pii'].sum()} / {len(df)} ({pii_pct:.1f}%)")
print(f"  Emails:    {df['has_email'].sum()}")
print(f"  CPFs:      {df['has_cpf'].sum()}")
print(f"  Telefones: {df['has_phone'].sum()}")

fig, ax = plt.subplots(figsize=(5, 3))
pd.Series({"Com PII": df["has_pii"].sum(), "Sem PII": (~df["has_pii"]).sum()}).plot(
    kind="bar", ax=ax, color=["tomato", "mediumseagreen"], rot=0
)
ax.set_title("Tickets com e sem PII detectada")
ax.set_ylabel("Contagem")
plt.tight_layout()
plt.show()

## 6. Resumo estatístico

In [ ]:
summary = {
    "Total de tickets": len(df),
    "Categorias únicas": df["category"].nunique(),
    "Canais únicos": df["channel"].nunique(),
    "Tickets com PII": int(df["has_pii"].sum()),
    "% com PII": f"{pii_pct:.1f}%",
    "Comprimento médio (chars)": round(df["msg_len"].mean(), 1),
    "Classe majoritária": cat_counts.idxmax(),
    "Classe minoritária": cat_counts.idxmin(),
    "Nulos em message": int(df["message"].isna().sum()),
}
for k, v in summary.items():
    print(f"{k:<30} {v}")